# Buổi 10 — Notebook 1: Hồi quy Tuyến tính

**Mục tiêu:** Hiểu và tự cài đặt ba cách tìm tham số cho mô hình hồi quy tuyến tính:
1. **Phương trình chuẩn** (closed-form solution)
2. **Gradient Descent** (Batch GD)
3. **Stochastic Gradient Descent** (SGD)

**Dữ liệu:** `data/nha_dat.csv` — 320 mẫu giá nhà (triệu đồng) với 4 đặc trưng:
- `dien_tich`: diện tích (m²)
- `so_phong`: số phòng ngủ
- `kc_trung_tam`: khoảng cách đến trung tâm thành phố (km)
- `tuoi_nha`: tuổi nhà (năm)

**Ước tính thời gian:** ~90 phút

> **Cách làm:** Mở `src/hoi_quy.py` và `src/preprocessing.py`, hoàn thiện
> từng hàm có `raise NotImplementedError(...)`, rồi chạy lại ô bên dưới.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from src import preprocessing, hoi_quy


## 1. Khám phá dữ liệu (EDA)


In [ ]:
df = pd.read_csv('../data/nha_dat.csv')
print(f'Shape: {df.shape}')
df.head(8)


In [ ]:
df.describe().round(1)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
cols = ['dien_tich', 'so_phong', 'kc_trung_tam', 'tuoi_nha']
for ax, col in zip(axes.flat, cols):
    ax.scatter(df[col], df['gia'], alpha=0.4, s=15)
    ax.set_xlabel(col)
    ax.set_ylabel('Giá (triệu đồng)')
plt.suptitle('Quan hệ đặc trưng — Giá nhà', fontsize=13)
plt.tight_layout()
plt.show()


> **Câu hỏi 1.1:** Đặc trưng nào có quan hệ tuyến tính rõ nhất với giá nhà?
> Đặc trưng nào có quan hệ nghịch chiều?


## 2. Chuẩn bị dữ liệu


In [ ]:
X = df[['dien_tich', 'so_phong', 'kc_trung_tam', 'tuoi_nha']].values
y = df['gia'].values
print(f'X: {X.shape}   y: {y.shape}')


### 2.1 Chia train / test

**Hàm cần hoàn thiện:** `preprocessing.chia_train_test()`


In [ ]:
X_train, X_test, y_train, y_test = preprocessing.chia_train_test(X, y)
print(f'Train: {X_train.shape[0]} mẫu  |  Test: {X_test.shape[0]} mẫu')


### 2.2 Chuẩn hóa đặc trưng

**Hàm cần hoàn thiện:** `preprocessing.chuan_hoa()`

> ⚠️ Chỉ **fit** scaler trên `X_train`. Nếu fit cả `X_test` → data leakage.


In [ ]:
scaler, X_tr_sc, X_te_sc = preprocessing.chuan_hoa(X_train, X_test)
print(f'Mean đặc trưng 0 (train scaled) : {X_tr_sc[:, 0].mean():.4f}  # nên ≈ 0')
print(f'Std  đặc trưng 0 (train scaled) : {X_tr_sc[:, 0].std():.4f}  # nên ≈ 1')


### 2.3 Thêm cột bias

Để viết `ŷ = Xw` mà không tách `w₀` (bias) riêng, ta thêm cột 1 vào đầu `X`.


In [ ]:
X_tr_b = hoi_quy.them_bias(X_tr_sc)
X_te_b = hoi_quy.them_bias(X_te_sc)
print(f'X_tr_b shape: {X_tr_b.shape}   (thêm 1 cột bias)')


## 3. Phương trình chuẩn (Normal Equation)

Giải tích trực tiếp:  **w* = (XᵀX)⁻¹ Xᵀy**

**Hàm cần hoàn thiện:** `hoi_quy.phuong_trinh_chuan()`


In [ ]:
w_ne = hoi_quy.phuong_trinh_chuan(X_tr_b, y_train)
print('w (Normal Equation):', np.round(w_ne, 2))

y_pred_ne = hoi_quy.du_doan(X_te_b, w_ne)
mse_ne = hoi_quy.tinh_mse(y_test, y_pred_ne)
print(f'MSE test (Normal Eq.): {mse_ne:.1f}')


> **Hàm cần hoàn thiện:** `hoi_quy.du_doan()` và `hoi_quy.tinh_mse()`


## 4. Gradient Descent

**Ý tưởng:** thay vì giải trực tiếp, ta *tự động tiến* về phía giảm loss:

```
w ← w − α · ∇w MSE
```

**Các hàm cần hoàn thiện theo thứ tự:**
1. `hoi_quy.tinh_mse()` (nếu chưa làm ở trên)
2. `hoi_quy.tinh_gradient()`
3. `hoi_quy.gradient_descent()`


In [ ]:
# Kiểm tra tinh_gradient trước khi chạy GD
w_test = np.zeros(X_tr_b.shape[1])
grad_test = hoi_quy.tinh_gradient(X_tr_b, y_train, w_test)
print('gradient shape:', grad_test.shape)    # phải là (5,)
print('gradient (5 phần tử đầu):', np.round(grad_test, 2))


In [ ]:
w_gd, loss_gd = hoi_quy.gradient_descent(
    X_tr_b, y_train, alpha=0.05, n_iter=1000
)

plt.figure(figsize=(7, 3))
plt.plot(loss_gd)
plt.xlabel('Vòng lặp')
plt.ylabel('MSE')
plt.title('Gradient Descent — Đường cong hội tụ')
plt.tight_layout()
plt.show()

y_pred_gd = hoi_quy.du_doan(X_te_b, w_gd)
mse_gd = hoi_quy.tinh_mse(y_test, y_pred_gd)
print(f'MSE test (GD): {mse_gd:.1f}')


> **Câu hỏi 4.1:** Nếu `alpha` quá lớn (ví dụ 0.5), điều gì xảy ra với đường cong loss?
> Hãy thử và giải thích.


## 5. Stochastic Gradient Descent (SGD)

SGD cập nhật trọng số sau **từng mẫu**, không phải toàn bộ batch.
Điều này giúp SGD:
- Nhanh hơn với dữ liệu lớn
- Dễ thoát khỏi cực trị địa phương hơn
- Nhưng đường loss dao động (noisy) hơn

**Hàm cần hoàn thiện:** `hoi_quy.sgd()`


In [ ]:
w_sgd, loss_sgd = hoi_quy.sgd(
    X_tr_b, y_train, alpha=0.01, n_epochs=50
)

# So sánh đường hội tụ GD vs SGD (lấy 50 điểm đầu của GD để so cùng số epoch)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_gd[:50], label='Batch GD', linewidth=2)
axes[0].plot(loss_sgd,      label='SGD',      linewidth=1, alpha=0.8)
axes[0].set_title('50 vòng đầu')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(loss_gd, label='Batch GD (1000 iter)', linewidth=2)
axes[1].set_title('GD 1000 vòng lặp')
axes[1].set_xlabel('Vòng lặp')
axes[1].set_ylabel('MSE')
axes[1].legend()

plt.tight_layout()
plt.show()

y_pred_sgd = hoi_quy.du_doan(X_te_b, w_sgd)
mse_sgd = hoi_quy.tinh_mse(y_test, y_pred_sgd)
print(f'MSE test (SGD): {mse_sgd:.1f}')


> **Câu hỏi 5.1:** Tại sao đường loss của SGD dao động nhiều hơn Batch GD?
> Làm thế nào để làm mịn đường này (giảm noise)?


## 6. So sánh với scikit-learn

scikit-learn dùng thuật toán tối ưu hiệu quả hơn để giải cùng bài toán.
Hãy so sánh kết quả.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lr = LinearRegression()
lr.fit(X_tr_sc, y_train)
y_pred_sk = lr.predict(X_te_sc)

print('--- Bảng so sánh MSE (test) ---')
print(f'  Normal Equation : {hoi_quy.tinh_mse(y_test, y_pred_ne):.1f}')
print(f'  Gradient Descent: {hoi_quy.tinh_mse(y_test, y_pred_gd):.1f}')
print(f'  SGD             : {hoi_quy.tinh_mse(y_test, y_pred_sgd):.1f}')
print(f'  sklearn LR      : {mean_squared_error(y_test, y_pred_sk):.1f}')
print()
print(f'R² sklearn: {r2_score(y_test, y_pred_sk):.4f}')


In [ ]:
# Biểu đồ giá trị thực vs dự đoán (sklearn)
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_sk, alpha=0.5, s=20)
mn, mx = y_test.min(), y_test.max()
plt.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Dự đoán lý tưởng')
plt.xlabel('Giá thực (triệu đồng)')
plt.ylabel('Giá dự đoán (triệu đồng)')
plt.title('Thực vs Dự đoán — sklearn LinearRegression')
plt.legend()
plt.tight_layout()
plt.show()


## 7. Câu hỏi tổng kết

1. Tại sao Normal Equation không được dùng khi tập dữ liệu có hàng triệu mẫu?
2. Khi nào nên chọn SGD thay vì Batch GD?
3. Nếu thêm đặc trưng mới (ví dụ `dien_tich²`), mô hình có còn là *tuyến tính* không?  
   Vẽ thêm cột `dien_tich_sq = dien_tich² ` và chạy lại — MSE có giảm không?

---
*Tiếp theo: `02_PhanLop_TuyenTinh.ipynb`*
